# Fallback 2: auditing evidence of covert price signals

Separate from the pricing experiments: programmed senders/recipients produce records; LLMs audit them.
No claim about spontaneous collusion or real-world detection. Offline review is complete. It uses its own
key, so it cannot spend the main study's money. No paid calls have been run locally.

Upload this notebook and `collusion-fallback2.bundle` to `MyDrive/collusion-sprint/`. Add a **separate** key
as `COLLUSION_F2_KEY` in Colab Secrets, with a non-resetting $10 limit. Run all performs offline checks → endpoint check →
20-case canary → 400-case held-out evaluation per family → tables and plots. Wrong answers remain data.
Four families: 1,680 successful calls before retries; attempt ceilings total 2,240. These are not dollar estimates.
The $10 limit on this key is the hard spending bound for F2.

All four families run concurrently, with four independent calls per family. Never give two runtimes the same
output directory. After disconnect, reconnect and Run all to resume saved replies; attempts do not reset.
Colab/Drive persistence still needs live validation. No manual pilot-enable flags.

Development and evaluation use different encodings AND policy rules. Evaluation is descriptive, 20 blocks
per family; correlated case types/views do not count as additional independent samples. The complete evidence
checks are live capability outcomes, not an accuracy threshold used to discard models. Citeable evidence strings
are stored but not automatically validated. See `paper/fallback2_case_spec.md` for scope and cutoffs.


In [ ]:
OFFLINE_PASSED = BUDGET_VERIFIED = False
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount('/content/drive')
SOURCE = '/content/drive/MyDrive/collusion-sprint/collusion-fallback2.bundle'
COMMIT = '014e76f91dc7064b3e1eb21792302d3de998238a'
REPO = Path('/content/collusion-fallback2')
OUT = Path('/content/drive/MyDrive/collusion-sprint/runs/fallback2_v1')
FAMILIES = ['deepseek41', 'qwen38', 'mistral3', 'gptoss120']

def sh(args, **kwargs):
    subprocess.run(args, check=True, **kwargs)

if not REPO.exists():
    sh(['git', 'clone', '-q', SOURCE, str(REPO)])
else:
    sh(['git', '-C', str(REPO), 'fetch', SOURCE])
sh(['git', '-C', str(REPO), 'checkout', '-q', COMMIT])
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sh(['bash', 'scripts/get_starter.sh'])
sh([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '-e', 'starter/util',
    '-e', 'starter/algorithmic-pricing-llms'])
os.environ['OPENROUTER_API_KEY'] = userdata.get('COLLUSION_F2_KEY')
OUT.mkdir(parents=True, exist_ok=True)
print('Pinned code:', COMMIT)


In [ ]:
OFFLINE_PASSED = False
sh([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'])
sh([sys.executable, '-m', 'pytest', '-q', 'tests/test_covert.py', 'tests/test_covert_notebook.py',
    'tests/test_report_figures.py'], env={**os.environ, 'MPLBACKEND': 'Agg'})
sh([sys.executable, 'scripts/check_covert_controls.py', '--out', str(OUT / 'controls')])
sh([sys.executable, '-m', 'sprint.covert_run', 'configs/covert_canary.toml',
    '--model-config', 'configs/main_pilot_qwen38.toml', '--stub', 'reference', '--out', str(OUT / 'offline')])
for family in FAMILIES:
    sh([sys.executable, '-m', 'sprint.covert_run', 'configs/covert_pilot.toml',
        '--model-config', f'configs/main_pilot_{family}.toml', '--dry-run'])
OFFLINE_PASSED = True


In [ ]:
BUDGET_VERIFIED = False
import urllib.request


# Metadata request only: no model call, key never printed or saved. F2 has its own $10 key;
# a larger limit means the wrong key, e.g. the main study's $100 one.
def check_key_budget(data, ceiling=10):
    limit, remaining = data.get('limit'), data.get('limit_remaining')
    if not isinstance(limit, (int, float)) or not 0 < limit <= ceiling or data.get('limit_reset') is not None:
        raise RuntimeError(f'Use the separate F2 key, with a non-resetting limit of at most ${ceiling}, then rerun.')
    if not isinstance(remaining, (int, float)) or remaining <= 0:
        raise RuntimeError(f'No remaining key allowance. Check spending; do not raise the ${ceiling} ceiling.')
    return {'key_limit_usd': limit, 'remaining_allowance_usd': remaining}

req = urllib.request.Request('https://openrouter.ai/api/v1/key',
    headers={'Authorization': 'Bearer ' + os.environ['OPENROUTER_API_KEY']})
with urllib.request.urlopen(req, timeout=30) as response:
    metadata = json.load(response)['data']
print(check_key_budget(metadata))
# Key allowance is a spending cap, not proof of sufficient prepaid account credit.
del metadata, req

BUDGET_VERIFIED = True


In [ ]:
from datetime import UTC, datetime

from sprint.covert_run import load_config


def endpoint_snapshot(config):
    model = config['model']
    url = 'https://openrouter.ai/api/v1/models/' + model.model + '/endpoints'
    with urllib.request.urlopen(url, timeout=30) as response:
        data = json.load(response)['data']
    endpoints = [e for e in data['endpoints'] if e.get('tag') == model.provider_route]
    if len(endpoints) != 1:
        raise RuntimeError(f'Pinned endpoint unavailable: {model.provider_route}')
    endpoint = endpoints[0]
    if endpoint.get('provider_name') != model.provider:
        raise RuntimeError('Endpoint provider identity changed')
    if model.quantization and endpoint.get('quantization') != model.quantization:
        raise RuntimeError('Endpoint quantization changed')
    needed = {'seed', 'temperature', 'max_tokens', 'response_format'}
    if model.reasoning_effort or model.reasoning_max_tokens:
        needed.add('reasoning')
    if not needed.issubset(endpoint.get('supported_parameters', [])):
        raise RuntimeError('Pinned endpoint no longer advertises the required parameters')
    signature = {k: endpoint.get(k) for k in
                 ('tag', 'provider_name', 'quantization', 'context_length', 'max_completion_tokens')}
    signature['supported_parameters'] = sorted(needed)
    signature['model'] = model.model
    anchor = OUT / (config['name'] + '.endpoint.json')
    with (OUT / (config['name'] + '.endpoint_history.jsonl')).open('a') as log:
        log.write(json.dumps({'time_utc': datetime.now(UTC).isoformat(), 'endpoint': endpoint}) + '\n')
    previous = json.loads(anchor.read_text()) if anchor.exists() else signature
    # Older anchors listed all advertised parameters; compare only those our requests use.
    previous['supported_parameters'] = sorted(needed.intersection(previous.get('supported_parameters', [])))
    if previous != signature:
        raise RuntimeError('Endpoint metadata changed during collection; retain logs and review before resuming')
    anchor.write_text(json.dumps(signature, indent=2) + '\n')



In [ ]:
if not globals().get('OFFLINE_PASSED') or not globals().get('BUDGET_VERIFIED'):
    raise RuntimeError('Offline controls and shared key budget must pass before model calls.')

from concurrent.futures import ThreadPoolExecutor, as_completed
from sprint.covert_analyze import launch_checks


def run_family(family):
    model_path = f'configs/main_pilot_{family}.toml'
    canary = load_config('configs/covert_canary.toml', model_path)
    pilot = load_config('configs/covert_pilot.toml', model_path)
    endpoint_snapshot(pilot)
    console = OUT / f'{family}.console.log'
    print(f'{family}: canary; progress -> {console}', flush=True)
    with console.open('a', buffering=1) as log:
        def command(stage, budget):
            sh([sys.executable, '-u', '-m', 'sprint.covert_run', f'configs/covert_{stage}.toml',
                '--model-config', model_path, '--out', str(OUT), '--budget', str(budget)],
               stdout=log, stderr=subprocess.STDOUT)
        command('canary', 60)
        checks = launch_checks(OUT / canary['name'])
        print(f'{family}: {checks}', flush=True)
        if not all(checks.values()):
            raise RuntimeError(f'{family}: transport checks failed; inspect {console}')
        command('pilot', 500)
        endpoint_snapshot(pilot)
    print(f'{family}: finished', flush=True)


failures = {}
with ThreadPoolExecutor(max_workers=4) as pool:
    jobs = {pool.submit(run_family, family): family for family in FAMILIES}
    for future in as_completed(jobs):
        family = jobs[future]
        try:
            future.result()
        except Exception as error:
            failures[family] = str(error)
            print(f'{family}: STOPPED: {error}', flush=True)
print('Family failures:', failures or 'none')
# The next cell exports partial results too; failures remain visible in accounting.


In [ ]:
from IPython.display import Image, display
from sprint.covert_analyze import export_tables, plot, summary

run_dirs = [OUT / load_config('configs/covert_pilot.toml', f'configs/main_pilot_{family}.toml')['name']
            for family in FAMILIES]
table = export_tables(run_dirs, OUT)
plot(table, OUT)
if table.empty:
    print('No evaluation cases started. Inspect console logs; no zero-error conclusion is available.')
else:
    display(table.drop_duplicates(['run', 'case_id']).groupby(['model', 'status']).size().rename('cases'))
    display(summary(table))
    for picture in sorted(OUT.glob('accuracy_*.png')):
        display(Image(filename=str(picture)))
from scripts import report_figures

figures = OUT / 'figures'
report_figures.run('f2', OUT, figures, 'F2: auditing evidence of covert price signals')
report_figures.run('diagrams', None, figures)
for picture in sorted(figures.glob('f2_baselines_*.png')):
    display(Image(filename=str(picture)))
print('Durable outputs:', OUT, '| figures:', figures)
if failures:
    raise RuntimeError('Some families stopped; partial results were exported. Inspect failures before resuming.')
